# Example usecases of scGPT-annotator

# 1. Prepare Data

# 2. Embed Data

# 3. Classify Data

# 3. Evaluate results


In [1]:
# Evaluate results
import os
from pathlib import Path
import scanpy as sc
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

file_path = r"C:\Users\annel\OneDrive\Documenten\Machine Learning\scGPT\scGPT-annotator\save\predicted_adata_20250414_02\predicted_adata_pred_results_20250414_002819.h5ad"

In [2]:
adata = sc.read(file_path)

In [3]:
print (adata)

AnnData object with n_obs × n_vars = 113899 × 26232
    obs: 'nCount_RNA', 'nFeature_RNA', 'Size_Factor', 'n.umi', 'scrublet_score', 'scrublet_call', 'sample', 'Ligation_barcode', 'RT_barcode', 'P7_barcode', 'P5_barcode', 'line', 'line_and_time.point', 'time.point', 'perc_mito_umi', 'monocle_cluster', 'sex_ontology_term_id', 'donor_id', 'organism_ontology_term_id', 'tissue_ontology_term_id', 'tissue_type', 'assay_ontology_term_id', 'disease_ontology_term_id', 'cell_type_ontology_term_id', 'self_reported_ethnicity_ontology_term_id', 'development_stage_ontology_term_id', 'suspension_type', 'is_primary_data', 'cluster', 'Xu_S1B_max_cell_type', 'cell_type', 'assay', 'disease', 'organism', 'sex', 'tissue', 'self_reported_ethnicity', 'development_stage', 'observation_joinid', 'pred_cell_type', 'pred_cell_type_prob_Schwann cell', 'pred_cell_type_prob_blood cell', 'pred_cell_type_prob_endodermal cell', 'pred_cell_type_prob_endothelial cell', 'pred_cell_type_prob_epidermal cell', 'pred_cell_typ

In [5]:
# Select the columns we want to display
display_cols = [
    'cell_type',  # True label
    'pred_cell_type',  # Predicted label
    'top1_type', 'top1_prob',
    'top2_type', 'top2_prob',
    'top3_type', 'top3_prob',
    'top4_type', 'top4_prob',
    'top5_type', 'top5_prob',
    'true_in_top_n'
]

# Create the table
results_df = adata.obs[display_cols]

# Format probabilities as percentages
prob_cols = ['top1_prob', 'top2_prob', 'top3_prob', 'top4_prob', 'top5_prob']
for col in prob_cols:
    results_df[col] = results_df[col].map('{:.1%}'.format)

# Display first few rows
print("\nPrediction Results:")
print("==================")
print(results_df.head())

# Show some statistics
correct_predictions = (results_df['cell_type'] == results_df['pred_cell_type']).mean()
in_top_n = results_df['true_in_top_n'].mean()

print("\nSummary Statistics:")
print(f"Correct predictions: {correct_predictions:.1%}")
print(f"True label in top 5: {in_top_n:.1%}")


Prediction Results:
                 cell_type              pred_cell_type  \
0         endothelial cell            endothelial cell   
1          mesodermal cell             mesodermal cell   
2  lateral mesodermal cell     lateral mesodermal cell   
3          endodermal cell             endodermal cell   
4            paraxial cell  splanchnic mesodermal cell   

                    top1_type top1_prob                     top2_type  \
0            endothelial cell    100.0%    splanchnic mesodermal cell   
1             mesodermal cell     27.0%                epidermal cell   
2     lateral mesodermal cell     36.0%  intermediate mesodermal cell   
3             endodermal cell     74.0%               epithelial cell   
4  splanchnic mesodermal cell     14.0%  intermediate mesodermal cell   

  top2_prob                     top3_type top3_prob             top4_type  \
0      0.0%  intermediate mesodermal cell      0.0%            blood cell   
1     19.0%                sensory ne

C:\Users\annel\AppData\Local\Temp\ipykernel_11772\362132593.py:19: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  results_df[col] = results_df[col].map('{:.1%}'.format)


In [6]:
# Create a clean summary DataFrame
summary_cols = [
    'cell_type',  # True label
    'pred_cell_type',  # Predicted label
    'top1_type', 'top1_prob',
    'top2_type', 'top2_prob',
    'top3_type', 'top3_prob',
    'top4_type', 'top4_prob',
    'top5_type', 'top5_prob',
    'true_in_top_n'
]

# Create the summary DataFrame
summary_df = adata.obs[summary_cols]

# Format probabilities as percentages
prob_cols = ['top1_prob', 'top2_prob', 'top3_prob', 'top4_prob', 'top5_prob']
for col in prob_cols:
    summary_df[col] = summary_df[col].map('{:.1%}'.format)

# Store in adata.uns for later use (e.g., visualization)
adata.uns['prediction_summary'] = {
    'results': summary_df.to_dict(),
    'statistics': {
        'accuracy': (summary_df['cell_type'] == summary_df['pred_cell_type']).mean(),
        'in_top_n': summary_df['true_in_top_n'].mean()
    }
}

# Display sample of results
print("\nPrediction Results Sample:")
print("=========================")
print(summary_df.head().to_string())

# Display statistics
print("\nSummary Statistics:")
print(f"Accuracy: {adata.uns['prediction_summary']['statistics']['accuracy']:.1%}")
print(f"True label in top 5: {adata.uns['prediction_summary']['statistics']['in_top_n']:.1%}")

C:\Users\annel\AppData\Local\Temp\ipykernel_11772\2269377735.py:19: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  summary_df[col] = summary_df[col].map('{:.1%}'.format)



Prediction Results Sample:
                 cell_type              pred_cell_type                   top1_type top1_prob                     top2_type top2_prob                     top3_type top3_prob             top4_type top4_prob                   top5_type top5_prob  true_in_top_n
0         endothelial cell            endothelial cell            endothelial cell    100.0%    splanchnic mesodermal cell      0.0%  intermediate mesodermal cell      0.0%            blood cell      0.0%             endodermal cell      0.0%           True
1          mesodermal cell             mesodermal cell             mesodermal cell     27.0%                epidermal cell     19.0%                sensory neuron     14.0%  primordial germ cell     11.0%             epithelial cell      9.0%           True
2  lateral mesodermal cell     lateral mesodermal cell     lateral mesodermal cell     36.0%  intermediate mesodermal cell     11.0%             somatic stem cell     10.0%        sensory neuron    

In [8]:
import numpy as np
import pandas as pd

def add_top_n_predictions_test(adata, n=5, pred_cell_col='pred_cell_type', cell_type_col='cell_type'):
    """
    Standalone version to test in a notebook.
    
    Args:
        adata: AnnData object with predictions
        n: Number of top predictions to store (default 5)
        pred_cell_col: Column name for predicted cell type
        cell_type_col: Column name for true cell type (if available)
        
    Returns:
        adata: Updated AnnData object
    """	
    # Get probability columns
    prob_cols = [col for col in adata.obs.columns if col.startswith(f'{pred_cell_col}_prob_')]
    
    if not prob_cols:
        print(f"No probability columns found with prefix '{pred_cell_col}_prob_'")
        print(f"Available columns: {list(adata.obs.columns)}")
        return adata
    
    print(f"Found {len(prob_cols)} probability columns")
    
    # Extract cell type names
    cell_types = [col.replace(f'{pred_cell_col}_prob_', '') for col in prob_cols]
    
    # Create probability matrix
    prob_matrix = np.zeros((adata.n_obs, len(cell_types)))
    for i, col in enumerate(prob_cols):
        prob_matrix[:, i] = adata.obs[col].values
    
    # Store full probability matrix in obsm
    adata.obsm['cell_type_probabilities'] = prob_matrix
    adata.uns['cell_type_probability_classes'] = cell_types
    
    # Get confidence score (probability of top prediction)
    top_probabilities = np.max(prob_matrix, axis=1)
    adata.obs['prediction_confidence'] = top_probabilities
    
    # For each cell, add top N predictions to obs
    for i in range(adata.n_obs):
        # Get indices of top N probabilities
        top_indices = np.argsort(prob_matrix[i])[-n:][::-1]
        
        # Add top N types and probabilities to obs
        for rank, idx in enumerate(top_indices):
            cell_type = cell_types[idx]
            probability = prob_matrix[i, idx]
            
            # Add as new columns (rank+1 to start numbering at 1)
            adata.obs.loc[adata.obs.index[i], f"top{rank+1}_type"] = cell_type
            adata.obs.loc[adata.obs.index[i], f"top{rank+1}_prob"] = probability
    
    # Calculate if true cell type is in top N predictions
    top_n_accuracy = None
    if cell_type_col in adata.obs.columns:
        in_top_n = []
        for i in range(adata.n_obs):
            true_type = adata.obs[cell_type_col].iloc[i]
            if pd.isna(true_type):
                in_top_n.append(False)
                continue
                
            top_types = [adata.obs[f"top{j+1}_type"].iloc[i] for j in range(n)]
            in_top_n.append(true_type in top_types)
        
        adata.obs['true_in_top_n'] = in_top_n
        valid_idx = ~adata.obs[cell_type_col].isna()
        top_n_accuracy = sum(in_top_n) / sum(valid_idx)
        print(f"Top-{n} accuracy: {top_n_accuracy:.4f}")
    
    # Create a summary DataFrame for adata.uns
    summary_cols = ['pred_cell_type', 'prediction_confidence']
    if cell_type_col in adata.obs.columns:
        summary_cols = [cell_type_col] + summary_cols
    
    for i in range(1, n+1):
        summary_cols.extend([f'top{i}_type', f'top{i}_prob'])
    
    if 'true_in_top_n' in adata.obs.columns:
        summary_cols.append('true_in_top_n')
    
    # Create a copy to avoid modifying the original
    summary_df = adata.obs[summary_cols].copy()
    
    # Format probabilities as percentages for display
    for prob_col in [f'top{i}_prob' for i in range(1, n+1)] + ['prediction_confidence']:
        if prob_col in summary_df.columns:
            summary_df[prob_col] = summary_df[prob_col].map('{:.1%}'.format)
    
    # Store summary in adata.uns
    adata.uns['prediction_summary'] = {
        'results': summary_df.head(1000).to_dict('records'),  # First 1000 cells for preview
        'statistics': {
            'accuracy': (adata.obs[cell_type_col] == adata.obs[pred_cell_col]).mean() 
                        if cell_type_col in adata.obs.columns else None,
            'top_n_accuracy': top_n_accuracy,
            'mean_confidence': float(np.mean(top_probabilities)),
            'high_confidence_cells': float(np.mean(top_probabilities > 0.9))  # % of cells with >90% confidence
        }
    }
    
    # Preview the results
    print("\nSummary statistics:")
    for k, v in adata.uns['prediction_summary']['statistics'].items():
        if v is not None:
            print(f"  {k}: {v:.4f}")
    
    print("\nTop predictions for first 5 cells:")
    preview_cols = ['prediction_confidence'] + [f'top{i}_type' for i in range(1, n+1)] + [f'top{i}_prob' for i in range(1, n+1)]
    display(adata.obs[preview_cols].head())
    
    return adata

# Example usage
# Just run this in your notebook:
adata = add_top_n_predictions_test(adata, n=5, pred_cell_col='pred_cell_type', cell_type_col='cell_type')

Found 17 probability columns
Top-5 accuracy: 0.8594

Summary statistics:
  accuracy: 0.4818
  top_n_accuracy: 0.8594
  mean_confidence: 0.4125
  high_confidence_cells: 0.0784

Top predictions for first 5 cells:


,prediction_confidence,top1_type,top2_type,top3_type,top4_type,top5_type,top1_prob,top2_prob,top3_prob,top4_prob,top5_prob
0,1.00,endothelial cell,splanchnic mesodermal cell,intermediate mesodermal cell,blood cell,endodermal cell,1.00,0.00,0.00,0.00,0.00
1,0.27,mesodermal cell,epidermal cell,sensory neuron,primordial germ cell,epithelial cell,0.27,0.19,0.14,0.11,0.09
2,0.36,lateral mesodermal cell,intermediate mesodermal cell,somatic stem cell,sensory neuron,epithelial cell,0.36,0.11,0.10,0.09,0.09
3,0.74,endodermal cell,epithelial cell,sensory neuron,primordial germ cell,splanchnic mesodermal cell,0.74,0.17,0.03,0.03,0.01
4,0.14,splanchnic mesodermal cell,intermediate mesodermal cell,sensory neuron,blood cell,mesodermal cell,0.14,0.13,0.12,0.08,0.07


In [4]:
print (adata.uns.keys())

dict_keys(['cell_type_colors', 'cell_type_probability_classes', 'citation', 'default_embedding', 'neighbors', 'sample_colors', 'schema_reference', 'schema_version', 'title', 'umap'])


In [5]:
display(adata.uns['prediction_summary'])

KeyError: 'prediction_summary'

In [6]:
from scGPT_classifier import scGPTAnnotator

c:\Users\annel\anaconda3\envs\scgpt_py39\lib\site-packages\scgpt\model\model.py:21: UserWarning: flash_attn is not installed
  warnings.warn("flash_attn is not installed")
c:\Users\annel\anaconda3\envs\scgpt_py39\lib\site-packages\scgpt\model\multiomic_model.py:19: UserWarning: flash_attn is not installed
  warnings.warn("flash_attn is not installed")


In [7]:
annotator = scGPTAnnotator(embedding_key='X_scGPT')

In [8]:
pred_n = annotator.add_top_n_predictions(adata)

Top-5 accuracy: 0.8594
Prediction data stored in adata.uns['prediction_data']
This contains exactly what you need: cell_type, pred_cell_type, and top predictions


In [10]:
display(adata.uns["prediction_data"])

,cell_type,pred_cell_type,prediction_confidence,top1_type,top1_prob,top2_type,top2_prob,top3_type,top3_prob,top4_type,top4_prob,top5_type,top5_prob,true_in_top_n
0,endothelial cell,endothelial cell,1.00,endothelial cell,1.00,splanchnic mesodermal cell,0.00,intermediate mesodermal cell,0.00,blood cell,0.00,endodermal cell,0.00,True
1,mesodermal cell,mesodermal cell,0.27,mesodermal cell,0.27,epidermal cell,0.19,sensory neuron,0.14,primordial germ cell,0.11,epithelial cell,0.09,True
2,lateral mesodermal cell,lateral mesodermal cell,0.36,lateral mesodermal cell,0.36,intermediate mesodermal cell,0.11,somatic stem cell,0.10,sensory neuron,0.09,epithelial cell,0.09,True
3,endodermal cell,endodermal cell,0.74,endodermal cell,0.74,epithelial cell,0.17,sensory neuron,0.03,primordial germ cell,0.03,splanchnic mesodermal cell,0.01,True
4,paraxial cell,splanchnic mesodermal cell,0.14,splanchnic mesodermal cell,0.14,intermediate mesodermal cell,0.13,sensory neuron,0.12,blood cell,0.08,mesodermal cell,0.07,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
166080,mesodermal cell,epithelial cell,0.23,epithelial cell,0.23,neural progenitor cell,0.11,endothelial cell,0.09,somatic stem cell,0.08,splanchnic mesodermal cell,0.07,False
166081,mesodermal cell,somatic stem cell,0.18,somatic stem cell,0.18,blood cell,0.15,mesodermal cell,0.09,epithelial cell,0.09,lateral mesodermal cell,0.08,True
166082,mesodermal cell,blood cell,0.19,blood cell,0.19,epithelial cell,0.14,splanchnic mesodermal cell,0.10,mesodermal cell,0.10,endothelial cell,0.07,True
166083,neuron,mesodermal cell,0.16,mesodermal cell,0.16,neuron,0.13,neural progenitor cell,0.10,endothelial cell,0.09,primordial germ cell,0.07,True
